# Tugas 4 — Validation dan Sampling pada UNSW-NB15
### Klasifikasi trafik jaringan: **Normal (0)** vs **Attack (1)**

Notebook ini merupakan kelanjutan **EDA → Preprocessing → Modelling & Evaluasi** menggunakan dataset Kaggle yang sama, yaitu `darkn3t/unsw-nb15-training-set-csv`.

**Tujuan pembelajaran:**
1. Melakukan **data validation**: memeriksa kolom, tipe label, missing value, duplikasi, dan distribusi kelas.
2. Menerapkan **random sampling, stratified sampling, random undersampling, dan random oversampling**.
3. Melakukan **model validation** melalui *stratified holdout* dan **Stratified K-Fold Cross-Validation**.
4. Membandingkan strategi sampling menggunakan skor yang mempertimbangkan kedua kelas.
5. Menjelaskan hasil berdasarkan **angka yang benar-benar keluar saat notebook dijalankan**.

> **Metode anti-data-leakage:** pemisahan data dilakukan **sebelum** imputasi, encoding, scaling, dan resampling. Saat cross-validation, semua tahap tersebut di-*fit* ulang hanya dalam fold training melalui `imblearn.pipeline.Pipeline`. Data validation/test **tidak** di-*resample*.

## A. Dasar teori singkat

**Validation (validasi)** adalah kegiatan memeriksa kelayakan data dan menguji apakah model dapat bekerja pada data yang belum pernah dipakai untuk melatihnya. Di sini ada tiga tingkat:

- **Data validation:** periksa label, format data, missing value, dan duplikasi.
- **Holdout validation:** pisahkan dataset menjadi **training, validation, dan testing**.
- **K-Fold Cross-Validation:** ulangi pelatihan pada beberapa pembagian training untuk mengukur kestabilan kinerja. `StratifiedKFold` menjaga proporsi label dalam setiap fold.

**Sampling** adalah memilih atau menyeimbangkan contoh data:

- **Random sampling:** memilih sejumlah data secara acak. Proporsi kelas bisa berubah.
- **Stratified sampling:** memilih secara acak **dalam setiap kelas** sehingga komposisi label tetap representatif.
- **Random undersampling:** mengurangi jumlah sampel **kelas mayoritas** di data training.
- **Random oversampling:** menambah salinan contoh **kelas minoritas** di data training.

**Asumsi yang harus diuji:** pada data UNSW-NB15 yang sebelumnya dipakai, kelas Attack justru lebih banyak daripada Normal. Jadi, jangan menganggap serangan selalu kelas minoritas. `RandomUnderSampler` / `RandomOverSampler` akan mengenali mayoritas/minoritas berdasarkan **data training aktual**.

**Metrik utama:** `Macro-F1` dan `Balanced Accuracy` memberi bobot lebih adil pada kedua kelas. `Recall Attack` memeriksa kemampuan menangkap serangan, sedangkan `Precision Attack` menggambarkan ketepatan alarm serangan. Akurasi saja bisa menyesatkan pada data tidak seimbang.

## B. Setup Google Colab / Kaggle

Jalankan sel dari atas ke bawah (**Runtime → Run all**). Default-nya notebook akan mencari dataset di `/kaggle/input/...`, lalu memakai `kagglehub` untuk mengunduh dataset di Colab.

Jika unduhan gagal, ubah `DATA_SOURCE = "manual"` lalu isi `MANUAL_DATA_DIR` dengan folder CSV milikmu. CPU sudah cukup, tidak perlu GPU.

In [ ]:
import sys, subprocess, json, warnings, shutil
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
import sklearn

try:
    import imblearn
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'imbalanced-learn'])
    import imblearn

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, balanced_accuracy_score, confusion_matrix,
                             ConfusionMatrixDisplay, classification_report,
                             roc_auc_score, make_scorer)
from imblearn.over_sampling import RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline

RANDOM_STATE = 42
DATA_SOURCE = 'kaggle'  # 'kaggle' / 'manual'
KAGGLE_HANDLE = 'darkn3t/unsw-nb15-training-set-csv'
KAGGLE_INPUT_PATH = Path('/kaggle/input/datasets/darkn3t/unsw-nb15-training-set-csv')
MANUAL_DATA_DIR = Path('/content')
CV_FOLDS = 3  # boleh menjadi 5; 3 lebih ramah untuk Colab CPU
SAMPLE_SIZE = 3000

WORK_DIR = Path('/content') if Path('/content').is_dir() else (Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd())
OUTPUT_DIR = WORK_DIR / 'Hasil_Validation_Sampling_UNSW_NB15'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Python', sys.version.split()[0], '| sklearn', sklearn.__version__, '| imblearn', imblearn.__version__)
print('Output:', OUTPUT_DIR)

## C. Muat dataset yang sama dengan tugas sebelumnya

**Penting:** nama CSV pada salah satu versi Kaggle bisa berupa `UNSW_NB15_testing-set.csv`, **namun** pada eksekusi tugas sebelumnya isinya **175.341 baris**. Nama file tidak dijadikan bukti bahwa isinya adalah partisi test resmi. Notebook mencatat **nama dan jumlah baris** yang benar-benar ditemukan.

- Jika hanya ada **satu CSV**, buat holdout sendiri: **64% train, 16% validation, 20% test** (sama seperti Tugas 3).
- Jika ada **dua CSV** train/test, gunakan testing asli sebagai holdout dan split training 80:20 untuk validation.

In [ ]:
if DATA_SOURCE == 'kaggle':
    candidates = [KAGGLE_INPUT_PATH, Path('/kaggle/input/unsw-nb15-training-set-csv')]
    found_folders = [p for p in candidates if p.is_dir()]
    if found_folders:
        dataset_root = found_folders[0]
    else:
        try:
            import kagglehub
        except ImportError:
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'kagglehub'])
            import kagglehub
        try:
            dataset_root = Path(kagglehub.dataset_download(KAGGLE_HANDLE))
        except Exception as exc:
            raise RuntimeError("Gagal mengunduh Kaggle. Ubah DATA_SOURCE='manual', unggah file CSV, dan coba lagi.") from exc
elif DATA_SOURCE == 'manual':
    dataset_root = MANUAL_DATA_DIR
else:
    raise ValueError("DATA_SOURCE harus 'kaggle' atau 'manual'")

if not dataset_root.is_dir():
    raise FileNotFoundError(f'Folder dataset tidak ditemukan: {dataset_root}')
csv_files = sorted(p for p in dataset_root.rglob('*.csv') if p.is_file() and OUTPUT_DIR not in p.parents)
print('Lokasi dataset:', dataset_root)
for p in csv_files:
    print(' -', p.name, '=>', round(p.stat().st_size/1e6, 2), 'MB')
if not csv_files:
    raise FileNotFoundError('Tidak menemukan .csv. Pastikan sumber data sudah benar.')

def normalized_stem(p):
    return p.stem.lower().replace('_', '').replace('-', '').replace(' ', '')

if len(csv_files) == 1:
    TRAIN_FILE, TEST_FILE = csv_files[0], None
else:
    train_files = [p for p in csv_files if 'train' in normalized_stem(p)]
    test_files  = [p for p in csv_files if 'test' in normalized_stem(p)]
    if len(csv_files) == 2 and len(train_files) == 1 and len(test_files) == 1:
        TRAIN_FILE, TEST_FILE = train_files[0], test_files[0]
    else:
        raise ValueError('Lebih dari satu CSV dengan nama ambigu. Gunakan folder khusus satu CSV atau pasangan train/test.')

print('\nSumber development:', TRAIN_FILE.name)
print('Sumber test terpisah:', TEST_FILE.name if TEST_FILE else 'Tidak tersedia; buat holdout baru')

## D. Data Validation: skema, label, nilai kosong, dan duplikasi

Data yang targetnya bukan `0` atau `1` dikeluarkan dan jumlahnya dilaporkan. Duplikasi **tidak otomatis dihapus**, karena kejadian trafik yang berulang bisa bermakna; jika ada duplikasi lintas split, akan dicatat sebagai keterbatasan.

In [ ]:
def read_binary_csv(path):
    df = pd.read_csv(path, encoding='utf-8-sig', low_memory=False)
    df.columns = df.columns.astype(str).str.replace('\ufeff', '', regex=False).str.strip()
    if df.columns.duplicated().any():
        raise ValueError(f'Kolom duplikat dalam {path.name}')
    if 'label' not in df:
        raise ValueError(f"Target 'label' tidak ada pada {path.name}")
    numeric_target = pd.to_numeric(df['label'], errors='coerce')
    valid = numeric_target.isin([0, 1])
    rejected = int((~valid).sum())
    df = df.loc[valid].copy().reset_index(drop=True)
    df['label'] = numeric_target[valid].to_numpy().astype('int8')
    if df['label'].nunique() != 2:
        raise ValueError(f'{path.name} tidak memuat kedua kelas 0 dan 1')
    return df, rejected

source_train, reject_train = read_binary_csv(TRAIN_FILE)
source_test, reject_test = read_binary_csv(TEST_FILE) if TEST_FILE else (None, 0)

quality_rows = []
for name, frame, rejected in [('Sumber utama',source_train,reject_train), ('Sumber test',source_test,reject_test)]:
    if frame is not None:
        quality_rows.append({'Sumber':name, 'Baris':len(frame),'Kolom':frame.shape[1],
                             'Label invalid dibuang':rejected,
                             'Missing cells':int(frame.isna().sum().sum()),
                             'Duplikat persis':int(frame.duplicated().sum()),
                             'Normal (0)':int((frame['label']==0).sum()),
                             'Attack (1)':int((frame['label']==1).sum())})
quality_report = pd.DataFrame(quality_rows)
display(quality_report)
print('\nContoh data:')
display(source_train.head(3))
assert set(source_train['label'].unique()) == {0, 1}

## E. Holdout Validation — pembagian train / validation / test

`stratify=label` digunakan agar distribusi Normal dan Attack tetap representatif. Sebelum model diuji, **test set dikunci secara konseptual**: jangan memilih strategi hanya berdasarkan metrik test.

In [ ]:
if source_test is None:
    dev_df, test_df = train_test_split(source_train, test_size=0.20, stratify=source_train['label'], random_state=RANDOM_STATE)
    split_method = 'satu_csv_64_16_20_stratified'
else:
    dev_df, test_df = source_train, source_test
    split_method = 'dua_csv_train_test_with_validation'
train_df, val_df = train_test_split(dev_df, test_size=0.20, stratify=dev_df['label'], random_state=RANDOM_STATE)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

split_report = pd.DataFrame([{'Split':name,'Jumlah':len(d),'Normal (0)':int((d.label==0).sum()),
                             'Attack (1)':int((d.label==1).sum()),
                             '% Attack':round(100*d.label.mean(),2)}
                            for name,d in [('Train',train_df),('Validation',val_df),('Test',test_df)]])
display(split_report)
assert all(set(x.label.unique()) == {0,1} for x in (train_df,val_df,test_df))

plt.figure(figsize=(8,4))
for name,frame in [('Train',train_df),('Validation',val_df),('Test',test_df)]:
    counts = frame.label.value_counts().reindex([0,1],fill_value=0)
    plt.bar([f'{name} Normal',f'{name} Attack'], counts.values)
plt.ylabel('Jumlah baris')
plt.title('Distribusi kelas setelah stratified holdout')
plt.xticks(rotation=25,ha='right')
plt.tight_layout(); plt.show()

## F. Sampling acak dan Stratified Sampling (demonstrasi)

Eksperimen ini **hanya menggunakan training set**. Dua sampel dengan ukuran sama dibandingkan supaya terlihat bagaimana stratifikasi menjaga proporsi kelas. Sampel demonstrasi ini **bukan** data yang dipakai untuk cross-validation, sehingga evaluasi tetap menggunakan seluruh training set.

In [ ]:
from sklearn.model_selection import train_test_split as sample_split
n_sample = min(SAMPLE_SIZE, len(train_df)//2)
if n_sample < 10:
    raise ValueError('Training set terlalu kecil untuk demonstrasi sampling')
random_sample = train_df.sample(n=n_sample, random_state=RANDOM_STATE)
_, stratified_sample = sample_split(train_df, test_size=n_sample, stratify=train_df['label'], random_state=RANDOM_STATE)

sample_report = pd.DataFrame([{'Data':name,'Jumlah':len(df), 'Normal (0)':int((df.label==0).sum()),
                               'Attack (1)':int((df.label==1).sum()),
                               '% Attack':round(100*df.label.mean(),2)}
                              for name,df in [('Train penuh',train_df),('Random sampling',random_sample),('Stratified sampling',stratified_sample)]])
display(sample_report)
print('Stratified sampling menjaga komposisi kelas berdasarkan label; random sampling biasa tidak memberi jaminan itu.')

## G. Persiapan fitur untuk validasi model

Gunakan kolom target `label`, dan **keluarkan** `id` serta `attack_cat` dari masukan model biner. `attack_cat` merupakan informasi kategori serangan dan bisa menyebabkan *target leakage*.

Imputasi, standardisasi dan one-hot encoding **belum di-fit** pada seluruh dataset. Semua akan berada dalam pipeline sehingga dipelajari hanya pada training fold selama cross-validation.

In [ ]:
TARGET = 'label'
EXCLUDED = {'label','id','attack_cat'}
feature_cols = [c for c in train_df.columns if c not in EXCLUDED]
for split_name, frame in [('Validation',val_df),('Test',test_df)]:
    missing = sorted(set(feature_cols) - set(frame.columns))
    if missing:
        raise ValueError(f'{split_name} kehilangan fitur: {missing}')
cat_cols = [c for c in feature_cols if c in {'proto','service','state'} or not pd.api.types.is_numeric_dtype(train_df[c])]
num_cols = [c for c in feature_cols if c not in cat_cols]

def sanitize_features(frame):
    f = frame[feature_cols].copy()
    for col in num_cols:
        f[col] = pd.to_numeric(f[col],errors='coerce').replace([np.inf,-np.inf],np.nan)
    for col in cat_cols:
        s = f[col].astype('string').str.strip()
        s = s.mask(s.isin(['','?','??','N/A','NA','NULL','null']))
        f[col] = s.astype(object).where(s.notna(),np.nan)
    return f

X_train, X_val, X_test = [sanitize_features(d) for d in (train_df,val_df,test_df)]
y_train, y_val, y_test = [d[TARGET].astype(int).reset_index(drop=True) for d in (train_df,val_df,test_df)]

all_empty = X_train.columns[X_train.isna().all()].tolist()
if all_empty:
    warnings.warn('Kolom kosong total pada training dibuang: '+str(all_empty))
    X_train=X_train.drop(columns=all_empty); X_val=X_val.drop(columns=all_empty); X_test=X_test.drop(columns=all_empty)
    num_cols=[c for c in num_cols if c not in all_empty]
    cat_cols=[c for c in cat_cols if c not in all_empty]

print('Fitur awal:', X_train.shape[1], '| Numerik:',len(num_cols),'| Kategorikal:',cat_cols)
print('Fitur yang tidak digunakan:',sorted(EXCLUDED.intersection(train_df.columns)))
print('Distribusi training:',y_train.value_counts().sort_index().to_dict())

## H. Sampling untuk ketidakseimbangan kelas — hanya data training

Tiga strategi yang dibandingkan:

1. **Tanpa resampling** (baseline): mempertahankan distribusi alami.
2. **Random undersampling**: memangkas kelas yang paling banyak.
3. **Random oversampling**: menambah contoh kelas yang paling sedikit melalui pengulangan.

Kelas mayoritas/minoritas dihitung dari label aktual, **bukan ditebak**. Saat cross-validation, sampler akan bekerja **di dalam training fold**. Ini mencegah data duplikat hasil oversampling bocor ke validation fold.

In [ ]:
counts_train = y_train.value_counts().sort_index()
majority_label = int(counts_train.idxmax())
minority_label = int(counts_train.idxmin())
name_map = {0:'Normal',1:'Attack'}
print('Mayoritas:', name_map[majority_label],int(counts_train.max()),'baris')
print('Minoritas:', name_map[minority_label],int(counts_train.min()),'baris')

# HANYA menampilkan perubahan jumlah label; validasi/model bekerja dengan pipeline di bawah.
indices = np.arange(len(y_train)).reshape(-1,1)
preview_rows=[]
preview_rows.append({'Strategi':'Tanpa sampling', 'Normal (0)':int(counts_train.get(0,0)), 'Attack (1)':int(counts_train.get(1,0))})
for name,sampler in [('Undersampling',RandomUnderSampler(random_state=RANDOM_STATE)),
                     ('Oversampling',RandomOverSampler(random_state=RANDOM_STATE))]:
    _, y_after = sampler.fit_resample(indices,y_train)
    vals = pd.Series(y_after).value_counts()
    preview_rows.append({'Strategi':name,'Normal (0)':int(vals.get(0,0)),'Attack (1)':int(vals.get(1,0))})
resampling_report = pd.DataFrame(preview_rows)
display(resampling_report)
resampling_report.set_index('Strategi')[['Normal (0)','Attack (1)']].plot(kind='bar',figsize=(8,4),rot=0)
plt.title('Efek resampling terhadap jumlah kelas (training saja)');plt.ylabel('Jumlah sampel');plt.tight_layout();plt.show()

## I. Stratified K-Fold Cross-Validation + pipeline bebas leakage

`StratifiedKFold(n_splits=3)` membagi **training set saja** menjadi tiga fold. Untuk setiap fold, pipeline menjalankan:

**Imputasi → scaling/one-hot encoding → [sampling jika dipilih] → Logistic Regression.**

Karena sampling ditempatkan di **dalam `imblearn.Pipeline`**, data pada fold validasi tidak pernah disampling dan tidak ikut menentukan median, skala, atau kategori encoder.

**Kenapa Logistic Regression?** Model ini merupakan baseline klasifikasi yang relatif hemat CPU dan dapat menjelaskan perbedaan teknik sampling. Tujuan tugas adalah membandingkan **strategi validasi dan sampling**, bukan mencari algoritma paling kompleks.

In [ ]:
def make_model(strategy):
    transforms=[]
    if num_cols:
        numeric_pipe = SkPipeline([('imputer',SimpleImputer(strategy='median')),
                                   ('scaler',StandardScaler())])
        transforms.append(('numerik',numeric_pipe,num_cols))
    if cat_cols:
        try:
            encoder = OneHotEncoder(handle_unknown='ignore',sparse_output=True)
        except TypeError:
            encoder = OneHotEncoder(handle_unknown='ignore',sparse=True)
        categorical_pipe = SkPipeline([('imputer',SimpleImputer(strategy='most_frequent')),
                                      ('onehot',encoder)])
        transforms.append(('kategori',categorical_pipe,cat_cols))
    preprocessor = ColumnTransformer(transforms,remainder='drop',sparse_threshold=1.0)
    steps=[('prep',preprocessor)]
    if strategy=='Undersampling':
        steps.append(('sampling',RandomUnderSampler(random_state=RANDOM_STATE)))
    elif strategy=='Oversampling':
        steps.append(('sampling',RandomOverSampler(random_state=RANDOM_STATE)))
    elif strategy!='Tanpa sampling':
        raise ValueError(f'Strategi tidak dikenal: {strategy}')
    steps.append(('model',LogisticRegression(solver='liblinear',max_iter=400,random_state=RANDOM_STATE)))
    return ImbPipeline(steps=steps)

strategies=['Tanpa sampling','Undersampling','Oversampling']
cv = StratifiedKFold(n_splits=CV_FOLDS,shuffle=True,random_state=RANDOM_STATE)
scoring={
    'macro_f1':make_scorer(f1_score,average='macro',zero_division=0),
    'balanced_accuracy':'balanced_accuracy',
    'recall_attack':make_scorer(recall_score,pos_label=1,zero_division=0),
}
cv_rows=[]
for strategy in strategies:
    print('Cross-validation:',strategy)
    scores=cross_validate(make_model(strategy), X_train, y_train, cv=cv,
                          scoring=scoring, n_jobs=1, error_score='raise')
    for fold in range(CV_FOLDS):
        cv_rows.append({'Strategi':strategy,'Fold':fold+1,
                        'Macro-F1':float(scores['test_macro_f1'][fold]),
                        'Balanced Accuracy':float(scores['test_balanced_accuracy'][fold]),
                        'Recall Attack':float(scores['test_recall_attack'][fold])})
cv_fold_results=pd.DataFrame(cv_rows)
cv_summary=(cv_fold_results.groupby('Strategi',as_index=False)
            .agg(Macro_F1_mean=('Macro-F1','mean'),Macro_F1_std=('Macro-F1','std'),
                 Balanced_Accuracy_mean=('Balanced Accuracy','mean'),
                 Recall_Attack_mean=('Recall Attack','mean'))
            .sort_values('Macro_F1_mean',ascending=False).reset_index(drop=True))
display(cv_fold_results.round(4))
display(cv_summary.round(4))

## J. Analisis kestabilan skor Cross-Validation

Rata-rata `Macro-F1` menunjukkan performa keseluruhan lintas fold. Standar deviasi membantu memeriksa apakah skor antar fold konsisten. Nilai yang lebih tinggi **tidak selalu berarti** layak digunakan tanpa mempertimbangkan false positive dan false negative.

In [ ]:
fig,ax=plt.subplots(figsize=(9,4))
ax.bar(cv_summary['Strategi'],cv_summary['Macro_F1_mean'],yerr=cv_summary['Macro_F1_std'],capsize=5)
ax.set_ylabel('Macro-F1 (0–1)')
ax.set_title(f'Perbandingan strategi sampling — {CV_FOLDS}-Fold CV')
ax.set_ylim(0,1)
plt.tight_layout();plt.show()
BEST_STRATEGY = str(cv_summary.iloc[0]['Strategi'])
print('Strategi terpilih berdasarkan rerata CV Macro-F1:',BEST_STRATEGY)

## K. Validasi akhir dengan holdout validation & test

Setelah strategi terpilih melalui **CV pada training**, model yang sama di-*fit* pada **seluruh training set**. Model lalu diuji pada **validation holdout** untuk konfirmasi hasil dan **test holdout** sekali sebagai evaluasi akhir.

> Jangan menggunakan skor test untuk memilih strategi sampling, threshold, atau hyperparameter. Jika ingin menyesuaikan strategi, kembali ke training/CV atau validation terlebih dahulu.

In [ ]:
best_model=make_model(BEST_STRATEGY)
best_model.fit(X_train,y_train)

def evaluate_holdout(name, features, target):
    pred=best_model.predict(features)
    proba=best_model.predict_proba(features)[:,1]
    row={'Dataset':name,
         'Accuracy':accuracy_score(target,pred),
         'Balanced Accuracy':balanced_accuracy_score(target,pred),
         'Macro-F1':f1_score(target,pred,average='macro',zero_division=0),
         'Precision Attack':precision_score(target,pred,pos_label=1,zero_division=0),
         'Recall Attack':recall_score(target,pred,pos_label=1,zero_division=0),
         'ROC-AUC':roc_auc_score(target,proba)}
    print('\n===', name,'===')
    print(classification_report(target,pred,target_names=['Normal','Attack'],digits=4,zero_division=0))
    return row,pred

val_metrics,y_pred_val=evaluate_holdout('Validation',X_val,y_val)
test_metrics,y_pred_test=evaluate_holdout('Test',X_test,y_test)
model_metrics=pd.DataFrame([val_metrics,test_metrics])
display(model_metrics.round(4))

fig,axes=plt.subplots(1,2,figsize=(10,4))
for ax,truth,pred,title in [(axes[0],y_val,y_pred_val,'Validation'),(axes[1],y_test,y_pred_test,'Test')]:
    ConfusionMatrixDisplay(confusion_matrix(truth,pred,labels=[0,1]),display_labels=['Normal','Attack']).plot(ax=ax,colorbar=False,values_format='d')
    ax.set_title(f'Confusion Matrix — {title}')
plt.tight_layout();plt.show()

## L. Pembahasan otomatis berdasarkan hasil eksekusi

Penjelasan di bawah dibuat berdasarkan skor yang dihitung notebook saat dijalankan, bukan angka perkiraan. Bandingkan **rata-rata CV** dengan **holdout test** untuk mengecek apakah kemampuan generalisasi konsisten.

In [ ]:
win=cv_summary.iloc[0]
other=cv_summary.iloc[-1]
attack_majority = name_map[majority_label]
s0=split_report.set_index('Split')
comment=f"""### Hasil dan interpretasi (otomatis)

**1. Data Validation.** Dataset utama memuat **{len(source_train):,}** baris dengan **{source_train.shape[1]}** kolom. Label tidak valid yang dikeluarkan: **{reject_train}** baris. Missing value awal: **{int(source_train.isna().sum().sum()):,}** sel. Jumlah duplikasi persis awal: **{int(source_train.duplicated().sum()):,}** baris.

**2. Holdout Validation.** Metode pemisahan: `{split_method}`. Terdapat **{len(train_df):,}** data training, **{len(val_df):,}** data validation, dan **{len(test_df):,}** data testing; pembagian di-*stratify* berdasarkan `label`.

**3. Sampling.** Di training, kelas mayoritas adalah **{attack_majority}** (**{int(counts_train.max()):,}**) dan minoritas adalah **{name_map[minority_label]}** (**{int(counts_train.min()):,}**). Undersampling mengurangi mayoritas; oversampling menggandakan contoh minoritas. Kedua teknik **tidak** diterapkan pada validation/test.

**4. Cross-Validation.** Strategi terbaik berdasarkan rerata **Macro-F1 pada {CV_FOLDS}-Fold CV** adalah **{BEST_STRATEGY}** dengan Macro-F1 **{win['Macro_F1_mean']:.4f} ± {win['Macro_F1_std']:.4f}**. Perbedaan nilai antar fold digunakan untuk menilai stabilitas.

**5. Validasi dan testing.** Model terpilih memperoleh Macro-F1 **{val_metrics['Macro-F1']:.4f}** pada validation serta **{test_metrics['Macro-F1']:.4f}** pada testing. Balanced Accuracy testing **{test_metrics['Balanced Accuracy']:.4f}**, Recall Attack **{test_metrics['Recall Attack']:.4f}**, dan Precision Attack **{test_metrics['Precision Attack']:.4f}**.

**6. Kesimpulan.** Teknik sampling yang cocok harus dipilih berdasarkan evaluasi pada data yang tidak ikut di-*resample*. Kinerja pada dataset ini tidak boleh disimpulkan hanya dari accuracy; kesalahan mendeteksi trafik normal sebagai serangan (false positive) dan gagal mendeteksi serangan (false negative) juga perlu diperiksa melalui confusion matrix.

**Batasan:** eksperimen ini memakai Logistic Regression dan {CV_FOLDS}-Fold CV. Jika sumber Kaggle hanya menyediakan satu CSV, test set yang dibuat merupakan *holdout internal*, bukan partisi resmi 82.332 baris, sehingga nilainya tidak langsung dapat dibandingkan dengan eksperimen yang memakai partisi resmi.
"""
display(Markdown(comment))

## M. Ekspor hasil dan petunjuk pengumpulan

Folder `Hasil_Validation_Sampling_UNSW_NB15` menyimpan **ringkasan validasi data, sampling, cross-validation, holdout, dan pembahasan**. File ZIP disediakan untuk arsip; notebook `.ipynb` tetap menjadi berkas utama tugas.

**Saat mengumpulkan:** jalankan **Runtime → Run all**, simpan notebook dengan output, lalu unduh melalui **File → Download → Download .ipynb**. Pastikan dosen dapat melihat tabel/grafik hasil.

In [ ]:
quality_report.to_csv(OUTPUT_DIR/'data_validation.csv',index=False)
split_report.to_csv(OUTPUT_DIR/'holdout_split.csv',index=False)
sample_report.to_csv(OUTPUT_DIR/'random_vs_stratified_sampling.csv',index=False)
resampling_report.to_csv(OUTPUT_DIR/'class_balancing_sampling.csv',index=False)
cv_fold_results.to_csv(OUTPUT_DIR/'cv_per_fold.csv',index=False)
cv_summary.to_csv(OUTPUT_DIR/'cv_summary.csv',index=False)
model_metrics.to_csv(OUTPUT_DIR/'holdout_metrics.csv',index=False)
(OUTPUT_DIR/'pembahasan.md').write_text(comment,encoding='utf-8')
metadata={'dataset':'UNSW-NB15','kaggle_handle':KAGGLE_HANDLE if DATA_SOURCE=='kaggle' else None,
          'source_file':str(TRAIN_FILE),'separate_test_file':str(TEST_FILE) if TEST_FILE else None,
          'split_method':split_method,'random_state':RANDOM_STATE,'cv_folds':CV_FOLDS,
          'best_sampling_strategy_by_cv_macro_f1':BEST_STRATEGY,
          'target':'label','feature_exclusions':sorted(EXCLUDED),
          'n_samples':{'train':len(y_train),'validation':len(y_val),'test':len(y_test)},
          'timestamp_utc':datetime.now(timezone.utc).isoformat()}
(OUTPUT_DIR/'metadata.json').write_text(json.dumps(metadata,indent=2,ensure_ascii=False),encoding='utf-8')
zip_path = Path(shutil.make_archive(str(OUTPUT_DIR),'zip',root_dir=OUTPUT_DIR))
print('✅ Validasi dan sampling selesai. Artefak:')
for p in sorted(OUTPUT_DIR.iterdir()): print('-',p.name)
print('ZIP:',zip_path, f'({zip_path.stat().st_size/1e6:.2f} MB)')

## N. Kesimpulan materi — hal penting untuk presentasi

**Validation** memastikan model diuji pada contoh yang tidak dipakai untuk pelatihan, sedangkan **Sampling** mengatur cara memilih atau menyeimbangkan data. Praktik yang benar adalah **split terlebih dahulu**, lalu menerapkan semua transformasi dan resampling **hanya pada bagian training dari setiap fold**. Dengan begitu, hasil CV lebih mencerminkan generalisasi dibanding sampling seluruh dataset sebelum split.

Sampling seimbang **tidak selalu meningkatkan Macro-F1 atau Recall Attack**. Dampaknya harus dibuktikan dari hasil perbandingan yang baru kamu jalankan.

**Sumber konsep:** [UNSW-NB15 (UNSW Canberra)](https://research.unsw.edu.au/projects/unsw-nb15-dataset), [imbalanced-learn pipeline](https://imbalanced-learn.org/stable/references/pipeline.html), dan [StratifiedKFold (scikit-learn)](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedKFold.html).